In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zamamahmed211/skills")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skills' dataset.
Path to dataset files: /kaggle/input/skills


In [ ]:
!pip install pymupdf

In [ ]:
pip install pdfplumber

In [ ]:
import pdfplumber
import re
import json
import requests
import time
import base64

BASE_URL = "https://api.github.com/users/"

# -------------------- GitHub README --------------------
def get_readme(username, repo):
    try:
        url = f"https://api.github.com/repos/{username}/{repo}/readme"
        res = requests.get(url)
        if res.status_code != 200:
            return ""
        data = res.json()
        content = base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        return content[:2000]
    except:
        return ""

# -------------------- SKILL MATCH (NEW ADDITION) --------------------
def repo_skill_score(readme_text, required_skills):
    text = readme_text.lower()
    score = 0
    matched = []

    for skill in required_skills:
        if skill.lower() in text:
            score += 1
            matched.append(skill)

    return score, matched

# -------------------- GitHub Data (MODIFIED ONLY TOP REPO LOGIC) --------------------
def get_github_data(username, required_skills):
    try:
        user_url = BASE_URL + username
        repos_url = user_url + "/repos?per_page=30&sort=updated"
        events_url = user_url + "/events?per_page=30"

        repos = requests.get(repos_url).json()
        events = requests.get(events_url).json()

        repo_names = []
        total_stars = 0
        total_forks = 0
        languages = {}

        repo_scores = []   # 🔥 NEW

        for repo in repos:
            repo_names.append(repo["name"])
            total_stars += repo.get("stargazers_count", 0)
            total_forks += repo.get("forks_count", 0)

            lang_res = requests.get(repo["languages_url"]).json()
            for lang, count in lang_res.items():
                languages[lang] = languages.get(lang, 0) + count

            # -------------------- NEW LOGIC (SKILL BASED REPO SCORING) --------------------
            readme = get_readme(username, repo["name"])
            score, matched = repo_skill_score(readme, required_skills)

            repo_scores.append({
                "name": repo["name"],
                "score": score,
                "matched_skills": matched,
                "stars": repo.get("stargazers_count", 0)
            })

            time.sleep(0.2)

        sorted_langs = sorted(languages.items(), key=lambda x: x[1], reverse=True)
        top_languages = [lang for lang, _ in sorted_langs[:5]]

        push_events = sum(1 for e in events if e.get("type") == "PushEvent")

        # -------------------- MODIFIED TOP REPOS --------------------
        sorted_repos = sorted(
            repo_scores,
            key=lambda x: (x["score"], x["stars"]),
            reverse=True
        )

        top_repos = sorted_repos[:5]

        readme_text = ""
        for repo in top_repos[:3]:
            readme_text += get_readme(username, repo["name"]) + " "
            time.sleep(0.2)

        return {
            "github_languages": top_languages,
            "github_repos": repo_names[:10],
            "repo_count": len(repo_names),
            "star_count": total_stars,
            "fork_count": total_forks,
            "recent_activity": push_events,

            # 🔥 MODIFIED OUTPUT
            "top_repos": [
                {
                    "repo": r["name"],
                    "skill_score": r["score"],
                    "matched_skills": r["matched_skills"]
                }
                for r in top_repos
            ],

            "readme_text": readme_text.strip()
        }

    except Exception as e:
        print("Error:", e)
        return {
            "github_languages": [], "github_repos": [],
            "repo_count": 0, "star_count": 0, "fork_count": 0,
            "recent_activity": 0, "top_repos": [], "readme_text": ""
        }

# -------------------- USERNAME EXTRACTION --------------------
def extract_github_username(raw_link):
    match = re.search(r'github\.com/([A-Za-z0-9_.\-]+)', raw_link, re.I)
    return match.group(1) if match else raw_link.strip("/")

# -------------------- RESUME PARSER (UNCHANGED) --------------------
def extract_resume_data(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text += page.extract_text() + "\n"

    github_links = list(dict.fromkeys(re.findall(
        r'(?:https?://)?(?:www\.)?github\.com/[A-Za-z0-9_.\-]+(?:/[A-Za-z0-9_.\-]+)*',
        full_text, re.I
    )))

    linkedin = list(dict.fromkeys(re.findall(
        r'(?:https?://)?(?:www\.)?linkedin\.com/in/[A-Za-z0-9_.\-]+',
        full_text, re.I
    )))

    skills_raw = re.findall(r'Technical Skills(.*?)(?=Projects|Education|$)', full_text, re.S | re.I)
    skills = []
    if skills_raw:
        skills = re.split(r'[,\n|•]', skills_raw[0])
        skills = [s.strip() for s in skills if len(s.strip()) > 2]

    projects = re.findall(r'Projects(.*?)(?=Education|$)', full_text, re.S | re.I)
    project_titles = []
    if projects:
        project_titles = [p.strip() for p in re.split(r'\n', projects[0]) if len(p.strip()) > 3]

    return {
        "links": {"github": github_links, "linkedin": linkedin},
        "skills": skills,
        "project_titles": project_titles,
    }

# -------------------- PIPELINE --------------------
if __name__ == "__main__":
    pdf_path = "/ResumeLAurexia.pdf"

    # 🔥 REQUIRED SKILLS (JOB INPUT)
    required_skills = [
        "python", "fastapi", "docker",
        "kubernetes", "postgresql", "aws"
    ]

    resume_data = extract_resume_data(pdf_path)
    print("Resume data:\n", json.dumps(resume_data, indent=2))

    github_data = {}
    if resume_data["links"]["github"]:
        raw_link = resume_data["links"]["github"][0]
        username = extract_github_username(raw_link)

        print(f"\nFetching GitHub data for: {username}")

        github_data = get_github_data(username, required_skills)

        print("GitHub data:\n", json.dumps(github_data, indent=2))

    final = {**resume_data, "github_data": github_data}
    print("\nFinal combined:\n", json.dumps(final, indent=2))

Resume data:
 {
  "links": {
    "github": [
      "github.com/pingpongpulse"
    ],
    "linkedin": [
      "linkedin.com/in/a-s-s-s-koundinya-387a6b33a"
    ]
  },
  "skills": [
    "Languages: Python",
    "x86 Assembly Language",
    "MATLAB",
    "Core: Data Structures & Algorithms",
    "Operating Systems",
    "Database Concepts",
    "Software Testing",
    "Machine",
    "Learning (Training",
    "Ensembling & Deployment)",
    "Deep Learning",
    "Geospatial Analysis",
    "Signal Processing",
    "Frameworks & Libraries: Node.js",
    "Express.js",
    "React",
    "ReactFlow",
    "TailwindCSS",
    "OpenCV",
    "MediaPipe",
    "pyannote.audio",
    "librosa",
    "NumPy",
    "SciPy",
    "PyAV",
    "TensorFlow/Keras",
    "LightGBM",
    "XGBoost",
    "CatBoost",
    "scikit-learn",
    "Plotly Dash",
    "U-Net / Mask R-CNN / YOLO (CNN architectures)",
    "Tools & Platforms: Git",
    "Google Cloud",
    "Google Earth Engine",
    "Kaggle",
    "Hugging Face",
    